===CARGA DE DATOS===

In [1]:
import pandas as pd
import numpy as np

In [2]:
#Leer datos de csv, try-except por si existe un error en la lectura 
# de caracteres especiales

try:
    df = pd.read_csv('data/online_retail_II.csv')
except UnicodeDecodeError:
    df = pd.read_csv('data/online_retail_II.csv', encoding = 'Latin-1')

df.head()

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom


In [3]:
#Exploración inicial del dataframe

df.info()
print("\nValores nulos por columna:")
print(df.isnull().sum())
print("\nTransacciones con cantidad negativa:", (df['Quantity']<0).sum())

<class 'pandas.DataFrame'>
RangeIndex: 1067371 entries, 0 to 1067370
Data columns (total 8 columns):
 #   Column       Non-Null Count    Dtype  
---  ------       --------------    -----  
 0   Invoice      1067371 non-null  str    
 1   StockCode    1067371 non-null  str    
 2   Description  1062989 non-null  str    
 3   Quantity     1067371 non-null  int64  
 4   InvoiceDate  1067371 non-null  str    
 5   Price        1067371 non-null  float64
 6   Customer ID  824364 non-null   float64
 7   Country      1067371 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 65.1 MB

Valores nulos por columna:
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

Transacciones con cantidad negativa: 22950


In [ ]:
#Filtro de ID nulo

df = df.dropna(subset=['Customer ID'])

In [5]:
#Filtro de transacciones canceladas 

df=df[df['Quantity']>0] #Convierte el dataframe en todo el dataframe siempre 
                        #y cuando tenga una cantidad mayor a 0

df=df[~df['Invoice'].astype(str).str.startswith('C')]
#Elimina los que empiezan con 'C' de Credit Note por si alguno se 'escapa'

In [6]:
#Convertir columna InvoiceDate a tipo fecha

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')

print("Fechas que no se pudieron convertir:", df['InvoiceDate'].isna().sum())
df = df.dropna(subset=['InvoiceDate']) #Si no hay fecha, no lo tomamos en cuenta

Fechas que no se pudieron convertir: 0


In [7]:
df['TotalPrice']= df['Quantity'] * df['Price']

In [8]:
#Fecha de referencia

fecha_referencia = df['InvoiceDate'].max() + pd.Timedelta(days=1) #El cliente más reciente tendrá 1 día de recencia, para evitar valores nulos
print("Fecha de referencia para calcular Recencia:", fecha_referencia)

Fecha de referencia para calcular Recencia: 2011-12-10 12:50:00


In [9]:
#Cálculo de RFM (Recencia, Frecuencia, Monetario) por cliente

rfm = df.groupby('Customer ID').agg({
    'InvoiceDate': lambda fecha: (fecha_referencia - fecha.max()).days,
    'Invoice': 'nunique',
    'TotalPrice': 'sum'
})

rfm.columns = ['Recency', 'Frequency', 'Monetary']
rfm.head()

,Recency,Frequency,Monetary
Customer ID,,,
12346.0,326,12,77556.46
12347.0,2,8,5633.32
12348.0,75,5,2019.40
12349.0,19,4,4428.69
12350.0,310,1,334.40


In [10]:
print(rfm.shape)
rfm.describe()

(5881, 3)


,Recency,Frequency,Monetary
count,5881.000000,5881.000000,5881.000000
mean,201.457745,6.287196,3017.076888
std,209.474135,13.012879,14734.128619
min,1.000000,1.000000,0.000000
25%,26.000000,1.000000,347.800000
50%,96.000000,3.000000,897.620000
75%,380.000000,7.000000,2304.180000
max,739.000000,398.000000,608821.650000
